In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from river import datasets, linear_model, metrics

In [ ]:
dataset = datasets.synth.RandomRBFDrift(
    n_classes=5, n_features=10, n_centroids=5, change_speed=0.001, n_drift_centroids=2)
model = linear_model.SoftmaxRegression()

In [ ]:
n_samples = 5_000

X, Y = [], []
for x, y in dataset.take(n_samples):
    X.append([x[0], x[1]])
    Y.append(y)

X = np.array(X)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

idx = 200
ax1.scatter(X[:idx, 0], X[:idx, 1], c=Y[:idx], cmap="viridis", s=10)
ax1.set_title("Initial state")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2.scatter(X[-idx:, 0], X[-idx:, 1], c=Y[-idx:], cmap="viridis", s=10)
ax2.set_title("Final state")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

plt.show()

In [ ]:
set_sizes = []
accuracies = []
metric = metrics.Accuracy()

for i, (x, y) in enumerate(dataset.take(10_000)):
    y_pred_proba = model.predict_proba_one(x)

    threshold = 0.9
    sorted_probs = sorted(y_pred_proba.items(),
                          key=lambda item: item[1], reverse=True)

    conformal_set = []
    cumulative_prob = 0.0
    for cls, prob in sorted_probs:
        conformal_set.append(cls)
        cumulative_prob += prob
        if cumulative_prob >= threshold:
            break
    set_sizes.append(len(conformal_set))

    if len(conformal_set) != 1 or i < 100:
        model.learn_one(x, y)

    y_pred = model.predict_one(x)
    if y_pred is not None:
        metric.update(y, y_pred)
        accuracies.append(metric.get())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(set_sizes, "o")
ax1.set_title("Set Size (Efficiency)")

ax2.plot(accuracies[500:])
ax2.set_title("Accuracy")

plt.show()